# ADAPT-VQE for H₂ — Adaptive Operator Pool ansatz

ADAPT-VQE (Grimsley et al., 2018) builds the ansatz iteratively by:
1. Computing gradients of all pool operators with respect to the current state
2. Adding the operator with the largest gradient magnitude to the ansatz
3. Re-optimizing variational parameters
4. Repeating until all gradients fall below a threshold

**Key advantage over fixed UCCSD:** Sparser circuits, fewer parameters, potentially
better convergence — the ansatz is adapted to the specific molecule.

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)

from qiskit_algorithms import VQE, AdaptVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock
from qiskit.primitives import Estimator

print("All imports OK")

## Step 2 — H₂ Molecular Hamiltonian (equilibrium geometry)

In [ ]:
molecule = """
H 0.0 0.0 0.0
H 0.735 0.0 0.0
"""

driver = PySCFDriver(atom=molecule, basis="sto3g")
problem = driver.run()

num_spatial_orbitals = problem.num_spatial_orbitals
num_particles = problem.num_particles
nuclear_repulsion = problem.nuclear_repulsion_energy()

hamiltonian = problem.hamiltonian
second_q_op = hamiltonian.second_q_op()

mapper = JordanWignerMapper()
qubit_op = mapper.map(second_q_op)

print(f"Spatial orbitals: {num_spatial_orbitals}")
print(f"Qubits:           {qubit_op.num_qubits}")
print(f"Nuclear repulsion: {nuclear_repulsion:.6f} Ha")

## Step 3 — Exact Classical Baseline

In [ ]:
exact_matrix = qubit_op.to_matrix()
eigenvalues, _ = np.linalg.eigh(exact_matrix)
total_exact_energy = eigenvalues[0] + nuclear_repulsion

print(f"Exact ground state energy: {total_exact_energy:.12f} Ha")
print(f"Literature reference:      ~-1.137270 Ha")

## Step 4 — Hartree-Fock Reference

In [ ]:
initial_state = HartreeFock(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
)
print(f"Hartree-Fock initial state: {initial_state.num_qubits} qubits")

## Step 5 — Build the Operator Pool

The operator pool contains all single and double excitation operators.
ADAPT-VQE computes gradients for each and picks the largest one at each step.

In [ ]:
uccsd_pool = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
    excitations=[1, 2],
)

pool_ops, _ = uccsd_pool.excitation_ops()
print(f"Pool size: {len(pool_ops)} operators")
print(f"Pool operators: {pool_ops}")

## Step 6 — ADAPT-VQE Run

In [ ]:
estimator = Estimator()
optimizer = COBYLA(maxiter=500)

adapt_vqe = AdaptVQE(
    estimator=estimator,
    ansatz=None,
    optimizer=optimizer,
    initial_operator_pool=pool_ops,
    threshold=1e-6,
)

print("Running ADAPT-VQE...")
result_adapt = adapt_vqe.compute_minimum_eigenvalue(qubit_op)

adapt_energy = result_adapt.eigenvalue.real + nuclear_repulsion
adapt_error = abs(adapt_energy - total_exact_energy)

print(f"\nADAPT-VQE ground state energy: {adapt_energy:.12f} Ha")
print(f"Exact ground state energy:      {total_exact_energy:.12f} Ha")
print(f"Error gap:                      {adapt_error:.12f} Ha")
print(f"Operators added to pool:        {len(result_adapt.eigenvalue_history)}")

## Step 7 — Standard UCCSD-VQE (for comparison)

In [ ]:
ansatz = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
    excitations=[1, 2],
)

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=500),
)

print("Running standard UCCSD-VQE...")
result_uccsd = vqe.compute_minimum_eigenvalue(qubit_op)

uccsd_energy = result_uccsd.eigenvalue.real + nuclear_repulsion
uccsd_error = abs(uccsd_energy - total_exact_energy)
uccsd_params = ansatz.num_parameters

print(f"\nUCCSD-VQE ground state energy: {uccsd_energy:.12f} Ha")
print(f"Exact ground state energy:      {total_exact_energy:.12f} Ha")
print(f"Error gap:                     {uccsd_error:.12f} Ha")
print(f"Number of parameters:           {uccsd_params}")

## Step 8 — ADAPT-VQE Convergence: Energy per Step

In [ ]:
adapt_energy_history = [
    ev + nuclear_repulsion
    for ev in result_adapt.eigenvalue_history
]
uccsd_energy_history = [
    ev + nuclear_repulsion
    for ev in result_uccsd.eigenvalue_history
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(adapt_energy_history, 'b-o', markersize=5, label='ADAPT-VQE')
axes[0].axhline(y=total_exact_energy, color='red', linestyle='--',
                label=f'Exact ({total_exact_energy:.6f} Ha)')
axes[0].set_xlabel('ADAPT Iteration', fontsize=11)
axes[0].set_ylabel('Energy (Hartree)', fontsize=11)
axes[0].set_title('ADAPT-VQE Energy Convergence', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(uccsd_energy_history, 'g-s', markersize=3, label='UCCSD-VQE')
axes[1].axhline(y=total_exact_energy, color='red', linestyle='--',
                label=f'Exact ({total_exact_energy:.6f} Ha)')
axes[1].set_xlabel('COBYLA Iteration', fontsize=11)
axes[1].set_ylabel('Energy (Hartree)', fontsize=11)
axes[1].set_title('UCCSD-VQE Energy Convergence', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vqe_h2/adapt_convergence.png', dpi=150)
plt.show()

print(f"ADAPT converged in {len(adapt_energy_history)} steps")
print(f"UCCSD converged in {len(uccsd_energy_history)} iterations")

## Step 9 — Parameter Count Comparison

ADAPT-VQE uses only the operators that contribute significantly,
resulting in a much sparser ansatz than the full UCCSD.

In [ ]:
adapt_params = len(adapt_energy_history)

fig, ax = plt.subplots(figsize=(8, 5))
methods = ['ADAPT-VQE', 'UCCSD-VQE']
params = [adapt_params, uccsd_params]
energies = [adapt_energy, uccsd_energy]

bars = ax.bar(methods, params, color=['tab:blue', 'tab:green'], width=0.4)
ax.set_ylabel('Number of Parameters', fontsize=12)
ax.set_title('Ansatz Sparsity: ADAPT-VQE vs UCCSD-VQE', fontsize=13)
for bar, p in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(p), ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(params) * 1.2)
plt.tight_layout()
plt.savefig('vqe_h2/parameter_comparison.png', dpi=150)
plt.show()

print(f"ADAPT-VQE parameters: {adapt_params} (sparsity: {100*(1 - adapt_params/uccsd_params):.1f}% reduction)")
print(f"UCCSD-VQE parameters: {uccsd_params}")

## Step 10 — Final Results Comparison Table

In [ ]:
print("=" * 70)
print(f"{'RESULTS COMPARISON':^70}")
print("=" * 70)
print(f"{'Method':<20} {'Energy (Ha)':<20} {'Error (Ha)':<15} {'Params':<10}")
print("-" * 70)
print(f"{'Exact (classical)':<20} {total_exact_energy:<20.12f} {'0':<15} {'N/A':<10}")
print(f"{'ADAPT-VQE':<20} {adapt_energy:<20.12f} {adapt_error:<15.12f} {adapt_params:<10}")
print(f"{'UCCSD-VQE':<20} {uccsd_energy:<20.12f} {uccsd_error:<15.12f} {uccsd_params:<10}")
print("=" * 70)
print(f"\nADAPT-VQE sparsity gain: {adapt_params} vs {uccsd_params} parameters")
print(f"Energy improvement: ADAPT is {abs(adapt_energy - total_exact_energy):.2e} Ha from exact")

## Step 11 — Energy-Parameter Pareto Plot

Shows the trade-off between ansatz complexity (parameter count) and final energy quality.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter([uccsd_params], [uccsd_energy], s=150, color='tab:green',
           zorder=5, label='UCCSD-VQE')
ax.scatter([adapt_params], [adapt_energy], s=150, color='tab:blue',
           zorder=5, label='ADAPT-VQE')
ax.axhline(y=total_exact_energy, color='red', linestyle='--', linewidth=1.5,
           label=f'Exact ({total_exact_energy:.6f} Ha)')

ax.annotate(f'UCCSD\n{uccsd_params} params', (uccsd_params, uccsd_energy),
            textcoords='offset points', xytext=(10, -5), fontsize=10)
ax.annotate(f'ADAPT\n{adapt_params} params', (adapt_params, adapt_energy),
            textcoords='offset points', xytext=(10, 5), fontsize=10)

ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Final Energy (Hartree)', fontsize=12)
ax.set_title('Energy-Parameter Pareto: ADAPT-VQE vs UCCSD', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vqe_h2/energy_pareto.png', dpi=150)
plt.show()